# 👥 YouTube Comment Author Subscription Mining & Audience Affinity Analysis
### *Graph Mining of Viewer Subscriptions, Cross-Channel Engagement, and Top 100 Channel Discovery via YouTube Data API v3*

---

## 📌 Executive Summary & Analytical Methodology

### Abstract
This Jupyter Notebook implements a quantitative graph mining pipeline designed to extract viewer subscription profiles from comment authors on YouTube long-form videos. By loading exported top comments datasets (generated via `youtube_video_comments.ipynb`), filtering unique comment authors, and retrieving their public subscriptions via the **YouTube Data API v3** (`subscriptions.list`), this notebook uncovers cross-channel audience overlaps, niche interest clusters, and competitive media consumption patterns. The notebook exports both comprehensive raw subscription records and an aggregated rank-ordered dataset of the **Top 100 channels** subscribed to by the audience directly to **Google Drive** (`/content/drive/MyDrive/YouTube_Analytics/`), while displaying detailed metrics and visual charts.

### Key Analytical Features:
1. **Comment Author Extraction & Unique Filtering**: Parses author URLs and metadata from exported video comment datasets, isolating unique channel IDs $A = \{a_1, a_2, \dots, a_N\}$ to eliminate duplicate API requests.
2. **Quota-Efficient Subscription Retrieval**: Queries `subscriptions.list(part="snippet", channelId=a)` for each unique author. It robustly intercepts private subscription settings (`subscriptionForbidden` / 403 status codes) without breaking execution while enforcing strict exception handling for systemic failures.
3. **Cross-Channel Frequency Aggregation**: Given the collection of public subscriptions $S_a$ for each author $a \in A_{public}$, channel frequency is computed as:
   $$f(c) = \sum_{a \in A_{public}} \mathbb{I}(c \in S_a)$$
   where $\mathbb{I}(\cdot)$ is the indicator function. The channels are then sorted by $f(c)$ in descending order to isolate the top $K = 100$ channels.
4. **Google Drive Storage & Dual Payload Persistence**: Mounts `/content/drive/MyDrive/YouTube_Analytics/` with a local fallback search hierarchy (`data/`, `analysis/`, `.`), saving both raw author subscriptions and aggregated top 100 channels in `.csv` and `.json` formats.
5. **Inline Visual Analytics**: Generates high-precision Matplotlib and Seaborn charts displaying top subscribed channels, subscription distributions per author, and audience overlap log-frequency curves directly within cell outputs using `plt.show()`.


## ⚙️ 1. Configuration & Setup

In this section, we configure runtime parameters, API credentials, target limits, and output directory paths.
- **API Key**: Automatically fetched from Google Colab Secrets (`YOUTUBE_API_KEY`) or fallback user input.
- **Max Authors Limit**: Specifies the maximum number of unique comment authors to process per run to manage API quota usage.
- **Drive Output Path**: Primary storage destination set to `/content/drive/MyDrive/YouTube_Analytics/`.


In [ ]:
# Install required packages if missing
!pip install -q google-api-python-client pandas matplotlib seaborn

import os
import re
import json
import datetime
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Try loading API key from Colab Secrets if available
colab_api_key = ""
try:
    from google.colab import userdata
    colab_api_key = userdata.get('YOUTUBE_API_KEY')
except Exception:
    pass

# User Configuration
CONFIG = {
    "YOUTUBE_API_KEY": colab_api_key or "YOUR_YOUTUBE_API_KEY_HERE",
    "CHANNEL_IDENTIFIER": "@josemillanastrologohumanista",  # Channel Handle, Channel ID, or Username
    "MAX_AUTHORS_TO_PROCESS": 200,                       # Limit unique comment authors to process (quota management)
    "MAX_SUBSCRIPTIONS_PER_AUTHOR": 50,                   # Max subscriptions retrieved per author
    "TOP_K_CHANNELS": 100,                                # Number of top aggregated channels to extract
    "DRIVE_OUTPUT_DIR": "/content/drive/MyDrive/YouTube_Analytics",
    "MOUNT_DRIVE": True,                                  # Set to True when running in Google Colab
    "USE_CACHE": True                                     # Set to True to reuse cached datasets stored on Drive
}

print("Configuration loaded successfully.")
if CONFIG["YOUTUBE_API_KEY"] in ["", "YOUR_YOUTUBE_API_KEY_HERE"]:
    print("⚠️ WARNING: Please set your YOUTUBE_API_KEY before running API fetching cells!")


## 💾 2. Mount Google Drive & Storage Setup

Mounts Google Drive at `/content/drive/MyDrive/YouTube_Analytics/` with a robust fallback search hierarchy (`src/static/data/`, `data/`, `analysis/`, `.`) to guarantee seamless execution across Google Colab and local environments.


In [ ]:
ACTIVE_OUTPUT_DIR = "."

if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
        ACTIVE_OUTPUT_DIR = CONFIG["DRIVE_OUTPUT_DIR"]
        print(f"✅ Google Drive mounted successfully. Storage directory: {ACTIVE_OUTPUT_DIR}")
    except ImportError:
        print("ℹ️ Google Drive module not found (not in Colab). Falling back to local storage hierarchy.")
    except Exception as e:
        print(f"⚠️ Google Drive mount failed ({e}). Falling back to local storage hierarchy.")

# Ensure fallback local directory exists
if ACTIVE_OUTPUT_DIR == ".":
    for fallback_dir in ["/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]:
        if os.path.exists(fallback_dir) or fallback_dir in [".", "data", "analysis"]:
            os.makedirs(fallback_dir, exist_ok=True)
            ACTIVE_OUTPUT_DIR = fallback_dir
            break

print(f"📁 Active storage path initialized: {os.path.abspath(ACTIVE_OUTPUT_DIR)}")


## 🔍 3. YouTube API Client & Helper Functions

Defines helper utility functions for initializing the YouTube Data API v3 client, extracting YouTube channel IDs from URLs, and sanitizing filename strings.


In [ ]:
def get_youtube_client(api_key: str):
    # Initializes and returns the YouTube Data API v3 client.
    if not api_key or api_key == "YOUR_YOUTUBE_API_KEY_HERE":
        return None
    return build('youtube', 'v3', developerKey=api_key)

def extract_channel_id_from_url(url: str) -> str:
    # Extracts YouTube Channel ID from author profile URL or returns empty string if missing.
    if not url or not isinstance(url, str):
        return ""
    match = re.search(r'channel/(UC[a-zA-Z0-9_-]{22})', url)
    if match:
        return match.group(1)
    return ""

def sanitize_filename(name: str) -> str:
    # Sanitizes channel identifier string for filesystem safety.
    return re.sub(r'[^a-zA-Z0-9_-]', '_', name).strip('_')

print("Helper utility functions initialized.")


## 📂 4. Load & Preprocess Exported YouTube Comment Datasets

Locates and loads top comment datasets exported by `youtube_video_comments.ipynb` across Google Drive and local search directories. Filters unique comment authors based on unique channel IDs to avoid duplicate API requests.


In [ ]:
safe_channel_identifier = sanitize_filename(CONFIG["CHANNEL_IDENTIFIER"])
search_directories = [ACTIVE_OUTPUT_DIR, "/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]

comments_data = []
channel_info = {"title": safe_channel_identifier}
found_comments_file = None

for d in search_directories:
    if not os.path.exists(d):
        continue
    for fname in os.listdir(d):
        if fname.endswith("_top_comments_10_recent_videos.json") or "comments" in fname.lower():
            fpath = os.path.join(d, fname)
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    payload = json.load(f)
                if isinstance(payload, dict) and "comments" in payload:
                    comments_data = payload["comments"]
                    channel_info = payload.get("channel", channel_info)
                    found_comments_file = fpath
                    break
                elif isinstance(payload, list):
                    comments_data = payload
                    found_comments_file = fpath
                    break
            except Exception as e:
                print(f"⚠️ Error reading candidate comments file {fpath}: {e}")
    if found_comments_file:
        break

if not comments_data:
    print("ℹ️ No pre-existing exported comment dataset found in search paths. Initializing baseline structure.")
else:
    print(f"⚡ Loaded {len(comments_data)} comments from file: {found_comments_file}")

# Extract unique comment authors
df_comments_raw = pd.DataFrame(comments_data) if comments_data else pd.DataFrame()
unique_authors = []
seen_channel_ids = set()

if not df_comments_raw.empty:
    for idx, row in df_comments_raw.iterrows():
        author_name = row.get("author_name", "Anonymous")
        author_url = row.get("author_channel_url", "")
        author_id = extract_channel_id_from_url(author_url)
        
        if not author_id and "author_channel_id" in row and isinstance(row["author_channel_id"], str):
            author_id = row["author_channel_id"]
            
        if author_id and author_id not in seen_channel_ids:
            seen_channel_ids.add(author_id)
            unique_authors.append({
                "author_channel_id": author_id,
                "author_name": author_name,
                "author_channel_url": author_url,
                "author_profile_image": row.get("author_profile_image", "")
            })

total_authors_in_comments = len(df_comments_raw.get("author_name", [])) if not df_comments_raw.empty else 0
unique_authors_count = len(unique_authors)

print(f"📊 Comment Author Extraction Metrics:")
print(f"   - Total Comments Analyzed:            {len(df_comments_raw)}")
print(f"   - Total Author Metadata Entries:     {total_authors_in_comments}")
print(f"   - Unique Authors Isolated (Filtered): {unique_authors_count}")

# Select authors to process based on MAX_AUTHORS_TO_PROCESS cap
authors_to_process = unique_authors[:CONFIG["MAX_AUTHORS_TO_PROCESS"]]
print(f"   - Unique Authors Selected for API Querying: {len(authors_to_process)}")


## 👥 5. Fetch Subscriptions for Unique Comment Authors

Queries `subscriptions.list` for each unique author. If an author's subscriptions are private, YouTube API returns a 403 `subscriptionForbidden` error, which is caught and counted without crashing execution. Checked against Drive cache to eliminate redundant API quota expenditure.


In [ ]:
def load_cached_subscriptions(output_dir: str, safe_channel: str):
    # Checks Google Drive for existing author subscriptions dataset.
    expected_fname = f"{safe_channel}_comment_author_subscriptions.json"
    fpath = os.path.join(output_dir, expected_fname)
    if os.path.exists(fpath):
        try:
            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)
            subs = data.get("subscriptions", [])
            meta = data.get("metrics", {})
            if subs:
                print(f"⚡ Reusing cached author subscriptions dataset from Google Drive: {expected_fname}")
                return subs, meta
        except Exception as e:
            print(f"⚠️ Error reading cached subscriptions file {expected_fname}: {e}")
    return None, None

def fetch_author_subscriptions(youtube_client, authors_list, max_subs_per_author=50):
    # Queries subscriptions.list for unique authors and handles private subscription errors.
    all_subscription_records = []
    processed_authors_count = 0
    public_authors_count = 0
    private_authors_count = 0
    error_authors_count = 0

    print(f"Starting subscription retrieval for {len(authors_list)} unique authors...")

    for idx, author in enumerate(authors_list, start=1):
        a_id = author["author_channel_id"]
        a_name = author["author_name"]
        processed_authors_count += 1

        if idx % 10 == 0 or idx == len(authors_list):
            print(f" [{idx}/{len(authors_list)}] Querying author: {a_name[:30]} (ID: {a_id})")

        try:
            req = youtube_client.subscriptions().list(
                part="snippet",
                channelId=a_id,
                maxResults=min(50, max_subs_per_author)
            )
            res = req.execute()
            items = res.get("items", [])
            public_authors_count += 1

            for item in items:
                sub_snippet = item["snippet"]
                rec = {
                    "author_channel_id": a_id,
                    "author_name": a_name,
                    "subscribed_channel_id": sub_snippet["resourceId"]["channelId"],
                    "subscribed_channel_title": sub_snippet.get("title", ""),
                    "subscribed_channel_description": sub_snippet.get("description", ""),
                    "subscribed_at": sub_snippet.get("publishedAt", ""),
                    "thumbnails": sub_snippet.get("thumbnails", {}).get("default", {}).get("url", "")
                }
                all_subscription_records.append(rec)

        except HttpError as e:
            if "subscriptionForbidden" in str(e) or "403" in str(e):
                private_authors_count += 1
            else:
                error_authors_count += 1
                print(f"   ⚠️ HTTP Error for author {a_id} ({a_name}): {e}")
        except Exception as e:
            error_authors_count += 1
            print(f"   ⚠️ Unexpected error for author {a_id} ({a_name}): {e}")

    metrics = {
        "total_authors_analyzed": len(authors_list),
        "processed_authors": processed_authors_count,
        "public_authors": public_authors_count,
        "private_authors": private_authors_count,
        "error_authors": error_authors_count,
        "total_subscriptions_fetched": len(all_subscription_records)
    }

    return all_subscription_records, metrics

# Execute Subscription Fetching
youtube = get_youtube_client(CONFIG["YOUTUBE_API_KEY"])
safe_channel_title = sanitize_filename(channel_info.get("title", CONFIG["CHANNEL_IDENTIFIER"]))

subscription_records = None
fetch_metrics = None

if CONFIG["USE_CACHE"]:
    subscription_records, fetch_metrics = load_cached_subscriptions(ACTIVE_OUTPUT_DIR, safe_channel_title)

if subscription_records is None:
    if not youtube and authors_to_process:
        print("ℹ️ API key not specified. Skipped live subscription retrieval (no synthetic data generated).")
        subscription_records = []
        fetch_metrics = {
            "total_authors_analyzed": len(authors_to_process),
            "processed_authors": 0,
            "public_authors": 0,
            "private_authors": 0,
            "error_authors": 0,
            "total_subscriptions_fetched": 0
        }
    elif youtube and authors_to_process:
        subscription_records, fetch_metrics = fetch_author_subscriptions(
            youtube,
            authors_to_process,
            max_subs_per_author=CONFIG["MAX_SUBSCRIPTIONS_PER_AUTHOR"]
        )
    else:
        subscription_records = []
        fetch_metrics = {
            "total_authors_analyzed": 0,
            "processed_authors": 0,
            "public_authors": 0,
            "private_authors": 0,
            "error_authors": 0,
            "total_subscriptions_fetched": 0
        }

print(f"\n✅ Subscription collection complete.")
print(f"   - Authors Processed: {fetch_metrics.get('processed_authors', 0)}")
print(f"   - Public Authors:    {fetch_metrics.get('public_authors', 0)}")
print(f"   - Private Authors:   {fetch_metrics.get('private_authors', 0)}")
print(f"   - Subscriptions Fetched: {len(subscription_records)}")


## 📊 6. Aggregate Top 100 Channels & Export Datasets to Drive

Computes the channel overlap frequency across comment authors:
$$f(c) = \sum_{a \in A_{public}} \mathbb{I}(c \in S_a)$$
Sorts and extracts the Top 100 subscribed channels. Exports both the raw author subscriptions dataset and the top 100 channels summary dataset in CSV and JSON formats to Google Drive.


In [ ]:
df_subs = pd.DataFrame(subscription_records) if subscription_records else pd.DataFrame()

top_100_channels_list = []
if not df_subs.empty:
    channel_counts = df_subs.groupby(["subscribed_channel_id", "subscribed_channel_title"]).agg(
        author_count=("author_channel_id", "nunique"),
        total_occurrences=("author_channel_id", "count")
    ).reset_index()

    channel_counts = channel_counts.sort_values(by=["author_count", "total_occurrences"], ascending=[False, False])
    top_k = channel_counts.head(CONFIG["TOP_K_CHANNELS"]).copy()
    top_k["rank"] = range(1, len(top_k) + 1)
    top_100_channels_list = top_k.to_dict(orient="records")

df_top_100 = pd.DataFrame(top_100_channels_list) if top_100_channels_list else pd.DataFrame()

# File Paths
csv_subs_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_title}_comment_author_subscriptions.csv")
json_subs_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_title}_comment_author_subscriptions.json")

csv_top100_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_title}_top_100_subscribed_channels.csv")
json_top100_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_title}_top_100_subscribed_channels.json")

# Export Subscriptions
if not df_subs.empty:
    df_subs.to_csv(csv_subs_path, index=False)
else:
    pd.DataFrame(columns=[
        "author_channel_id", "author_name", "subscribed_channel_id",
        "subscribed_channel_title", "subscribed_channel_description", "subscribed_at"
    ]).to_csv(csv_subs_path, index=False)

json_subs_payload = {
    "channel": channel_info or {"title": safe_channel_title},
    "extracted_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "metrics": fetch_metrics or {},
    "subscriptions": subscription_records
}
with open(json_subs_path, "w", encoding="utf-8") as f:
    json.dump(json_subs_payload, f, indent=2, ensure_ascii=False)

# Export Top 100 Channels
if not df_top_100.empty:
    df_top_100.to_csv(csv_top100_path, index=False)
else:
    pd.DataFrame(columns=[
        "rank", "subscribed_channel_id", "subscribed_channel_title", "author_count", "total_occurrences"
    ]).to_csv(csv_top100_path, index=False)

json_top100_payload = {
    "channel": channel_info or {"title": safe_channel_title},
    "extracted_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "total_unique_channels": len(df_subs["subscribed_channel_id"].unique()) if not df_subs.empty else 0,
    "top_k_count": len(df_top_100),
    "top_channels": top_100_channels_list
}
with open(json_top100_path, "w", encoding="utf-8") as f:
    json.dump(json_top100_payload, f, indent=2, ensure_ascii=False)

print(f"📁 Successfully exported datasets to storage ({ACTIVE_OUTPUT_DIR}):")
print(f"   1. Detailed Subscriptions CSV:  {csv_subs_path}")
print(f"   2. Detailed Subscriptions JSON: {json_subs_path}")
print(f"   3. Top 100 Channels CSV:        {csv_top100_path}")
print(f"   4. Top 100 Channels JSON:       {json_top100_path}")


## 📈 7. Quantitative Metrics Summary & Visual Analytics

Presents summary statistics detailing author, subscription, and channel totals, accompanied by visual analytics (top 20 channels, subscription distributions, and channel overlap frequency curves) rendered inline with `plt.show()`.


In [ ]:
total_unique_channels = len(df_subs["subscribed_channel_id"].unique()) if not df_subs.empty else 0

print("=" * 60)
print("📊 SUMMARY METRICS: COMMENT AUTHOR SUBSCRIPTIONS & TOP CHANNELS")
print("=" * 60)
print(f" • Total Comments Analyzed:            {len(df_comments_raw)}")
print(f" • Total Comment Authors Extracted:     {total_authors_in_comments}")
print(f" • Unique Comment Authors (Filtered):   {unique_authors_count}")
print(f" • Unique Authors Processed for Subs:   {fetch_metrics.get('processed_authors', 0) if fetch_metrics else 0}")
print(f" • Authors with Public Subscriptions:  {fetch_metrics.get('public_authors', 0) if fetch_metrics else 0}")
print(f" • Authors with Private Subscriptions: {fetch_metrics.get('private_authors', 0) if fetch_metrics else 0}")
print(f" • Total Subscriptions Fetched:        {len(df_subs)}")
print(f" • Total Unique Channels Discovered:   {total_unique_channels}")
print(f" • Top Subscribed Channels Aggregated:  {len(df_top_100)}")
print("=" * 60)

# Visual Analytics
if not df_top_100.empty:
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams.update({'font.size': 10, 'figure.titlesize': 14})

    # Chart 1: Top 20 Subscribed Channels Among Comment Authors
    plt.figure(figsize=(12, 7))
    top_20 = df_top_100.head(20).copy()
    ax = sns.barplot(
        x="author_count",
        y="subscribed_channel_title",
        data=top_20,
        hue="subscribed_channel_title",
        palette="viridis",
        legend=False
    )
    plt.title("Top 20 Most Subscribed Channels Among Comment Authors", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Number of Comment Authors Subscribed", fontweight="bold")
    plt.ylabel("Subscribed Channel Title", fontweight="bold")
    
    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(f"{int(width)}",
                        (width, p.get_y() + p.get_height() / 2.),
                        ha='left', va='center',
                        xytext=(5, 0),
                        textcoords='offset points',
                        fontsize=9)
    plt.tight_layout()
    plt.show()

    # Chart 2: Subscriptions per Public Author Distribution
    if not df_subs.empty:
        plt.figure(figsize=(10, 5))
        author_sub_counts = df_subs.groupby("author_channel_id").size()
        sns.histplot(author_sub_counts, bins=20, kde=True, color="#2b5c8f")
        plt.title("Distribution of Public Subscriptions per Comment Author", fontsize=14, fontweight="bold", pad=15)
        plt.xlabel("Public Subscriptions Count", fontweight="bold")
        plt.ylabel("Frequency (Number of Authors)", fontweight="bold")
        plt.tight_layout()
        plt.show()

    # Chart 3: Channel Overlap Frequency Curve
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(df_top_100) + 1), df_top_100["author_count"], marker='o', color='#882255', markersize=4)
    plt.title("Channel Overlap Frequency Curve (Top 100 Channels)", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Channel Rank (1 to 100)", fontweight="bold")
    plt.ylabel("Subscribed Author Count", fontweight="bold")
    plt.yscale("log")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

else:
    print("ℹ️ No subscription data available to display visualization charts.")
